### O que é SCAN (Soma de prefixos)
É um algoritimo que pega o vetor de entra e devolve o vetor de saida, no qual cada elemento do vetor de entrada tem o valor somado com o seus antecessores.

In [10]:
# Bibliotecas usadas
import numpy as np
import math

In [11]:
# Abordagem sequencial clássica - O(N)
def scan_sequencial(X):
    Y = np.zeros_like(X)
    soma = 0
    for i in range(len(X)):
        soma += X[i]
        Y[i] = soma
    return Y

In [12]:
vetor_entrada = [1]*10
vetor_saida = scan_sequencial(vetor_entrada).tolist()

print("Vetor de entrada: ", vetor_entrada)
print("Vetor de saida: ", vetor_saida)

Vetor de entrada:  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Vetor de saida:  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


### O Problema da Dependência de Dados

No código acima, perceba que para calcular o valor de Y[i], nós dependemos obrigatoriamente do valor previamente computado em Y[i-1]. Isso caracteriza uma dependência de dados verdadeira (RAW - Read After Write). No modelo de execução sequencial, isso força um tempo de processamento estritamente linear de complexidade O(N).
##### A Solução Paralela (Hillis-Steele)
Para resolver esse gargalo, usamos algoritmos de prefixos paralelos (PPAs), como o método de Hillis-Steele. A lógica consiste em transformar a dependência linear em uma estrutura de árvore de somas, onde as threads do processador cooperam em passos discretos de tempo.

A cada passo K (onde stride = 2^k):

* Cada thread i soma seu valor atual com o valor localizado na posição i - stride.
* Se o índice i for menor que o stride, a thread apenas mantém o seu valor.

Com essa quebra de dependência, o tempo de execução cai drasticamente para uma complexidade paralela de O(log (N))  passos, assumindo que possuímos processadores paralelos suficientes (como os milhares de núcleos presentes em uma GPU).


In [16]:
def simulacao_hillis_steele(X):
    A = np.array(X, dtype=np.int32)
    n = len(A)
    num_passos = int(math.ceil(math.log2(n))) #2^n = k

    print("=== VETOR DE ENTRADA INICIAL ===")
    print(f"Estado inicial: {A.tolist()}\n")

    for passo in range(num_passos):
        stride = 2**passo
        # Fazemos uma cópia para simular a sincronia de barreira
        estado_anterior = A.copy()

        print(f"--- PASSO {passo + 1} (Salto/Stride = {stride}) ---")

        for i in range(n):
            if i >= stride:
                A[i] = estado_anterior[i] + estado_anterior[i - stride]
                print(f" Thread {i:02d}: somou pos {i} ({estado_anterior[i]}) + pos {i-stride} ({estado_anterior[i-stride]}) = {A[i]}")
            else:
                print(f" Thread {i:02d}: manteve o valor = {A[i]}")

        print(f"Resultado do passo {passo + 1}: {A.tolist()}\n")

    print("=== PROCESSAMENTO FINALIZADO ===")
    print(f"Resultado Inclusivo Final: {A.tolist()}")


In [15]:
# Execução de teste
vetor_teste = [1, 1, 1, 1, 1, 1, 1, 1]
simulacao_hillis_steele(vetor_teste)

=== VETOR DE ENTRADA INICIAL ===
Estado inicial: [1, 1, 1, 1, 1, 1, 1, 1]

--- PASSO 1 (Salto/Stride = 1) ---
 Thread 00: manteve o valor = 1
 Thread 01: somou pos 1 (1) + pos 0 (1) = 2
 Thread 02: somou pos 2 (1) + pos 1 (1) = 2
 Thread 03: somou pos 3 (1) + pos 2 (1) = 2
 Thread 04: somou pos 4 (1) + pos 3 (1) = 2
 Thread 05: somou pos 5 (1) + pos 4 (1) = 2
 Thread 06: somou pos 6 (1) + pos 5 (1) = 2
 Thread 07: somou pos 7 (1) + pos 6 (1) = 2
Resultado do passo 1: [1, 2, 2, 2, 2, 2, 2, 2]

--- PASSO 2 (Salto/Stride = 2) ---
 Thread 00: manteve o valor = 1
 Thread 01: manteve o valor = 2
 Thread 02: somou pos 2 (2) + pos 0 (1) = 3
 Thread 03: somou pos 3 (2) + pos 1 (2) = 4
 Thread 04: somou pos 4 (2) + pos 2 (2) = 4
 Thread 05: somou pos 5 (2) + pos 3 (2) = 4
 Thread 06: somou pos 6 (2) + pos 4 (2) = 4
 Thread 07: somou pos 7 (2) + pos 5 (2) = 4
Resultado do passo 2: [1, 2, 3, 4, 4, 4, 4, 4]

--- PASSO 3 (Salto/Stride = 4) ---
 Thread 00: manteve o valor = 1
 Thread 01: manteve o va

### DESAFIO PARA O ALUNO

O algoritmo de Hillis-Steele padrão gera uma soma Inclusiva. No entanto, muitas aplicações paralelas requerem um Scan Exclusivo (onde cada posição de saída contém as somas das posições estritamente anteriores a ela, começando obrigatoriamente com o valor 0).

A maneira mais eficiente em hardware para transformar um Scan Inclusivo em um Scan Exclusivo é aplicar um deslocamento de dados para a direita (Shift Right) por um elemento e preencher o início com o valor 0.

__Tarefa__

Complete o código abaixo inserindo a lógica de soma inclusiva vetorizada em NumPy (para simular a execução concorrente por fatias) e o deslocamento necessário para convertê-lo em Scan Exclusivo.


In [22]:
def scan_exclusivo_paralelo(X):
    """
    Desenvolva um algoritmo de soma de prefixos exclusivo paralelo simulado.
    Dica: Utilize fatiamento do NumPy (slices) para aplicar somas paralelas.
    """
    A = np.array(X, dtype=np.int32)
    n = len(A)
    num_passos = int(math.ceil(math.log2(n)))

    # -------------------------------------------------------------
    # ETAPA 1: Implemente a Soma Inclusiva (Hillis-Steele)
    # Dica: No passo 'k', some a fatia A[stride:] com anterior[0:n-stride]
    # -------------------------------------------------------------
    for passo in range(num_passos):
        stride = 2**passo
        anterior = A.copy()

        # === SEU CÓDIGO DA ETAPA 1 AQUI ===
        for i in range(n):
            if i >= stride:
                A[i] = anterior[i] + anterior[i - stride]
                print(f"Vetor pos {i}: ({anterior[i]}) + pos {i-stride} ({anterior[i-stride]}) = {A[i]}")
            else:
                print(f"Vetor pos {i}: ({anterior[i]}) menor que stride, mantido")
        print()
        # ==================================

    # -------------------------------------------------------------
    # ETAPA 2: Conversão de Inclusivo para Exclusivo (Shift Right)
    # Crie o vetor resultante Y cheio de zeros e desloque os dados.
    # Ex: [1, 3, 6, 10] deve virar [0, 1, 3, 6]
    # -------------------------------------------------------------
    Y = np.zeros_like(A)

    # === SEU CÓDIGO DA ETAPA 2 AQUI ===
    Y[1:] = A[:-1]
    # ==================================

    return Y.tolist()

In [23]:
# --- TESTE DO ALUNO ---
entrada_exercicio = [3, 1, 7, 0, 4, 1, 6, 3]
resultado_esperado = [0, 3, 4, 11, 11, 15, 16, 22]

resultado_aluno = scan_exclusivo_paralelo(entrada_exercicio)

print("Entrada:            ", entrada_exercicio)
print("Resultado do Aluno: ", list(resultado_aluno))
print("Resultado Esperado: ", resultado_esperado)

if np.array_equal(resultado_aluno, resultado_esperado):
    print("\n EXCELENTE TRABALHO! Seu algoritmo de Scan Exclusivo funcionou corretamente!")
else:
    print("\n Ops, algo está incorreto. Verifique as fatias do array ou a lógica de deslocamento.")


Vetor pos 0: (3) menor que stride, mantido
Vetor pos 1: (1) + pos 0 (3) = 4
Vetor pos 2: (7) + pos 1 (1) = 8
Vetor pos 3: (0) + pos 2 (7) = 7
Vetor pos 4: (4) + pos 3 (0) = 4
Vetor pos 5: (1) + pos 4 (4) = 5
Vetor pos 6: (6) + pos 5 (1) = 7
Vetor pos 7: (3) + pos 6 (6) = 9

Vetor pos 0: (3) menor que stride, mantido
Vetor pos 1: (4) menor que stride, mantido
Vetor pos 2: (8) + pos 0 (3) = 11
Vetor pos 3: (7) + pos 1 (4) = 11
Vetor pos 4: (4) + pos 2 (8) = 12
Vetor pos 5: (5) + pos 3 (7) = 12
Vetor pos 6: (7) + pos 4 (4) = 11
Vetor pos 7: (9) + pos 5 (5) = 14

Vetor pos 0: (3) menor que stride, mantido
Vetor pos 1: (4) menor que stride, mantido
Vetor pos 2: (11) menor que stride, mantido
Vetor pos 3: (11) menor que stride, mantido
Vetor pos 4: (12) + pos 0 (3) = 15
Vetor pos 5: (12) + pos 1 (4) = 16
Vetor pos 6: (11) + pos 2 (11) = 22
Vetor pos 7: (14) + pos 3 (11) = 25

Entrada:             [3, 1, 7, 0, 4, 1, 6, 3]
Resultado do Aluno:  [0, 3, 4, 11, 11, 15, 16, 22]
Resultado Esperado: 